# APMODE NODE Backend on Colab

Run the hybrid mechanistic-NODE backend on a free Colab T4 GPU.

**Prerequisites:** Select a GPU runtime (Runtime → Change runtime type → T4 GPU).

What this notebook covers:
1. Install APMODE with GPU-enabled JAX
2. Fetch a public PK dataset and run NCA
3. Train a NODE model with `gpu_fast` mode
4. Run functional distillation (surrogate fitting + fidelity)
5. Compare NODE vs classical candidates

## 1. Install

In [ ]:
# Clone the repo
!git clone https://github.com/biostochastics/apmode.git
%cd apmode

In [ ]:
# Swap jax[cpu] for jax[cuda12] to use the Colab GPU
!sed -i 's/jax\[cpu\]/jax[cuda12]/g' pyproject.toml

# Install uv and sync dependencies
!pip install uv
!uv sync --all-extras

In [ ]:
# Verify JAX sees the GPU
import jax

devices = jax.devices()
print(f"JAX devices: {devices}")
print(f"JAX version: {jax.__version__}")

assert any(d.platform == "gpu" for d in devices), (
    "No GPU found. Go to Runtime -> Change runtime type -> T4 GPU, "
    "then restart and re-run. If using CUDA 11, change the sed command "
    "above to jax[cuda11]."
)

## 2. Fetch and profile a dataset

In [ ]:
from pathlib import Path

from apmode.data.datasets import fetch_dataset, list_datasets

# Show available datasets
for ds in list_datasets():
    print(f"  {ds.name:20s}  {ds.n_subjects:3d} subj  {ds.route:12s}  {ds.description[:60]}")

In [ ]:
# Fetch theophylline (12 subjects, oral, linear elim — good for a quick test)
DATA_DIR = Path("/tmp/apmode_data")
csv_path = fetch_dataset("theo_sd", DATA_DIR)
print(f"Dataset: {csv_path}")

In [ ]:
from apmode.data.ingest import ingest_csv
from apmode.data.profiler import profile_dataset
from apmode.data.nca import estimate_nca

df = ingest_csv(csv_path)
data_manifest = profile_dataset(df)
nca = estimate_nca(df)

print(f"Subjects: {data_manifest.n_subjects}")
print(f"Route: {data_manifest.route}")
print(f"Richness: {data_manifest.richness_category}")
print(f"\nNCA estimates:")
for k, v in nca.items():
    print(f"  {k}: {v:.4g}")

## 3. Define a NODE model in Formular

In [ ]:
from apmode.dsl.grammar import compile_dsl
from apmode.dsl.validator import validate_dsl

# NODE absorption: let the neural network learn the absorption process
node_spec = compile_dsl("""
model {
    absorption: NODE_Absorption(dim=4, constraint_template=monotone_increasing)
    distribution: OneCmt(V=30.0)
    elimination: Linear(CL=2.0)
    variability: {
        IIV(params=[CL, V], structure=diagonal)
    }
    observation: Combined(sigma_prop=0.1, sigma_add=0.5)
}
""")

# Validate for Discovery lane (NODE is not admissible in Submission)
issues = validate_dsl(node_spec, lane="discovery")
print(f"Validation issues: {issues if issues else 'none'}")
print(f"Model ID: {node_spec.model_id}")
print(f"Has NODE modules: {node_spec.has_node_modules()}")

## 4. Train the NODE model (GPU)

In [ ]:
import asyncio
import time

from apmode.backends.node_runner import NODERunner
from apmode.backends.node_trainer import TrainingConfig

WORK_DIR = Path("/tmp/apmode_node")

runner = NODERunner(
    work_dir=WORK_DIR,
    execution_mode="gpu_fast",
    training_config=TrainingConfig(
        epochs=200,
        learning_rate=1e-3,
        early_stop_patience=20,
    ),
)

t0 = time.perf_counter()

result = asyncio.get_event_loop().run_until_complete(
    runner.run(
        spec=node_spec,
        data_manifest=data_manifest,
        initial_estimates=nca,
        seed=42,
        data_path=csv_path,
    )
)

elapsed = time.perf_counter() - t0
print(f"\nTraining completed in {elapsed:.1f}s")
print(f"Converged: {result.converged}")
print(f"OFV (-2LL): {result.ofv:.2f}")
print(f"AIC: {result.aic:.2f}")
print(f"BIC: {result.bic:.2f}")
print(f"Init source: {result.initial_estimate_source}")
print(f"\nParameter estimates:")
for name, est in result.parameter_estimates.items():
    print(f"  {name}: {est:.4g}")

## 5. Functional distillation

NODE interpretability via learned sub-function visualization,
parametric surrogate fitting, and AUC/Cmax bioequivalence fidelity.

In [ ]:
from apmode.backends.node_distillation import (
    distill_node_candidate,
    visualize_sub_function,
)
from apmode.backends.node_ode import HybridPKODE, ODEConfig

# Re-build the trained model for distillation
# (the runner doesn't expose the model directly, so we reconstruct)
ode_config = runner._build_ode_config(node_spec, nca)

import jax
from apmode.backends.node_init import transfer_from_classical

key = jax.random.PRNGKey(42)
transfer_result = transfer_from_classical(
    ode_config,
    classical_estimates=nca,
    key=key,
    use_pretrained=True,
)
model = transfer_result.model

# Visualize the learned absorption sub-function
x_vals, y_vals = visualize_sub_function(
    model,
    n_points=100,
    conc_range=(0.01, 50.0),
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(x_vals, y_vals, "b-", linewidth=2, label="NODE learned function")
ax.set_xlabel("Concentration")
ax.set_ylabel("NODE output (rate)")
ax.set_title("NODE Absorption Sub-Function")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from apmode.backends.node_distillation import fit_parametric_surrogate, assess_fidelity

# Fit a parametric surrogate to the NODE output
surrogate = fit_parametric_surrogate(x_vals, y_vals)
print(f"Best surrogate: {surrogate.surrogate_type}")
print(f"Parameters: {surrogate.params}")
print(f"R-squared: {surrogate.r_squared:.4f}")

# Assess bioequivalence fidelity (AUC/Cmax within 80-125% GMR)
fidelity = assess_fidelity(model, surrogate, ode_config)
print(f"\nFidelity:")
print(f"  AUC GMR:  {fidelity.auc_gmr:.3f}  {'PASS' if fidelity.auc_pass else 'FAIL'}")
print(f"  Cmax GMR: {fidelity.cmax_gmr:.3f}  {'PASS' if fidelity.cmax_pass else 'FAIL'}")
print(f"  Overall:  {'PASS' if fidelity.overall_pass else 'FAIL'}")

## 6. Compare: classical vs NODE

In [ ]:
# Define a classical model for the same dataset
classical_spec = compile_dsl("""
model {
    absorption: FirstOrder(ka=1.0)
    distribution: OneCmt(V=30.0)
    elimination: Linear(CL=2.0)
    variability: {
        IIV(params=[CL, V, ka], structure=diagonal)
    }
    observation: Combined(sigma_prop=0.1, sigma_add=0.5)
}
""")

print(f"Classical model: {classical_spec.model_id}")
print(f"NODE model:      {node_spec.model_id}")
print()
print("To run classical estimation, you need R + nlmixr2 installed.")
print("On Colab, install R with: !apt-get install -y r-base")
print("Then install nlmixr2: !Rscript -e 'install.packages(\"nlmixr2\", repos=\"https://cloud.r-project.org\")'") 
print()
print("For now, here's the side-by-side of what the governance funnel sees:")
print(f"\n  NODE result:")
print(f"    OFV:       {result.ofv:.2f}")
print(f"    AIC:       {result.aic:.2f}")
print(f"    Converged: {result.converged}")
print(f"    Backend:   {result.backend}")

## 7. GPU vs CPU benchmark

In [ ]:
# Re-run on CPU for comparison
cpu_runner = NODERunner(
    work_dir=Path("/tmp/apmode_node_cpu"),
    execution_mode="cpu_deterministic",
    training_config=TrainingConfig(
        epochs=200,
        learning_rate=1e-3,
        early_stop_patience=20,
    ),
)

t0 = time.perf_counter()
cpu_result = asyncio.get_event_loop().run_until_complete(
    cpu_runner.run(
        spec=node_spec,
        data_manifest=data_manifest,
        initial_estimates=nca,
        seed=42,
        data_path=csv_path,
    )
)
cpu_elapsed = time.perf_counter() - t0

print(f"GPU time: {elapsed:.1f}s")
print(f"CPU time: {cpu_elapsed:.1f}s")
print(f"Speedup:  {cpu_elapsed / elapsed:.1f}x")
print()
print("Note: For 12-subject theophylline, GPU overhead may negate gains.")
print("GPU shines with larger datasets (30+ subjects) and higher NODE dims.")
print()
print(f"GPU OFV: {result.ofv:.2f}  |  CPU OFV: {cpu_result.ofv:.2f}")
print("(Values may differ slightly — gpu_fast is not bitwise reproducible.)")

## Notes

- **~50 subject limit:** The NODE backend loops over subjects in Python (not `jax.vmap`). GPU accelerates per-subject ODE solves but doesn't remove this ceiling.
- **Determinism:** `cpu_deterministic` gives bitwise reproducibility. `gpu_fast` does not.
- **Discovery lane only:** NODE models are admissible in Discovery and Translational Optimization lanes. They are never eligible for Submission lane — this is a hard governance rule.
- **Formular is the control surface:** All models, including NODE, are specified in the Formular language. The NODE backend cannot bypass the typed grammar.